In [2]:
print("Hello world")

Hello world


In [3]:
import sys

# Check if we are currently running inside Google Colab
if 'google.colab' in sys.modules:
    print("Colab environment detected. Installing dependencies and cloning repository...")
    # Major project dependencies used by this notebook and local modules.
    colab_requirements = [
        'numpy',
        'sentencepiece',
        'datasets',
        'jax[cpu]',
        'matplotlib',
        'ipykernel',
    ]

    !pip install -q {" ".join(colab_requirements)}
    
    # Run the shell command to clone
    !git clone https://github.com/krishnan1159/NMT.git
    
    # Add the cloned repo to the system path
    sys.path.append('/content/NMT')
    
else:
    print("Local environment detected. Skipping git clone.")

Colab environment detected. Installing dependencies and cloning repository...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 100.5 MB/s eta 0:00:0000:01
Cloning into 'NMT'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 37 (delta 14), reused 29 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 411.50 KiB | 15.83 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [4]:
if 'google.colab' not in sys.modules:
    %load_ext autoreload
    %autoreload 2
    %reload_ext autoreload

## Local classes
import importlib
from vocab import Tokenizer
from model_embedding import ModelEmbedding
from config import get_config
import rnn

importlib.reload(rnn)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


<module 'rnn' from '/content/NMT/rnn.py'>

In [5]:
from datasets import load_dataset
import jax
import jax.numpy as jp

In [6]:
ds = load_dataset("shiyue/chr_en", "parallel")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/7.55k [00:00<?, ?B/s]

parallel/train-00000-of-00001.parquet:   0%|          | 0.00/1.75M [00:00<?, ?B/s]

parallel/dev-00000-of-00001.parquet:   0%|          | 0.00/149k [00:00<?, ?B/s]

parallel/out_dev-00000-of-00001.parquet:   0%|          | 0.00/46.3k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/11639 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating out_dev split:   0%|          | 0/256 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating out_test split:   0%|          | 0/256 [00:00<?, ? examples/s]

In [7]:
train_set = list(ds["train"]["sentence_pair"])
chr_sentences = [sentence['chr'] for sentence in train_set]
eng_sentences = [sentence['en'] for sentence in train_set]

## Generate vocab for both source and target sentence.
cfg = get_config()
chr_vocab = Tokenizer(chr_sentences, "chr", "chr", "20000")
eng_vocab = Tokenizer(eng_sentences, "eng", "eng", "8000")
random_key = jax.random.key(0)
embedding = ModelEmbedding(chr_vocab.get_vocab_size(), eng_vocab.get_vocab_size(), cfg.embed_size, random_key)

In [8]:
print(eng_sentences[0])
a, b = eng_vocab.to_numpy([eng_sentences[1]])
print(b)
print(a.shape)
embedding.target_embed(a).shape

and by him every one that believeth is justified from all things, from which ye could not be justified by the law of Moses.
[29]
(1, 29)


(1, 29, 256)

In [13]:
#chr_sentences = chr_sentences[:]
#eng_sentences = eng_sentences[:]

chr_tokens, chr_lengths = chr_vocab.to_numpy(chr_sentences, add_bos=True, add_eos=True)
eng_tokens, eng_lengths = eng_vocab.to_numpy(eng_sentences, add_bos=True, add_eos=True)
print(eng_vocab.get_vocab_size())
print(cfg.num_epochs)

rnn.train(chr_tokens, chr_lengths, eng_tokens, eng_lengths, embedding, eng_vocab.get_vocab_size(), cfg)

INFO:rnn:JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


INFO:rnn:JAX backend: tpu
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/a

8000
1000


INFO:rnn:Loss at epoch 37/1000 is 6.297338962554932. Time taken to train = 0.0048421089999237665
INFO:rnn:Loss at epoch 38/1000 is 6.201882362365723. Time taken to train = 0.004758749000075113
INFO:rnn:Loss at epoch 39/1000 is 6.211759090423584. Time taken to train = 0.004754660000003241
INFO:rnn:Loss at epoch 40/1000 is 6.200720310211182. Time taken to train = 0.004738669999937883
INFO:rnn:Loss at epoch 41/1000 is 6.281065464019775. Time taken to train = 0.004824760000019523
INFO:rnn:Loss at epoch 42/1000 is 6.16034460067749. Time taken to train = 0.004888288999950419
INFO:rnn:Loss at epoch 43/1000 is 6.154165267944336. Time taken to train = 0.004764509999972688
INFO:rnn:Loss at epoch 44/1000 is 6.151645183563232. Time taken to train = 0.004741289999969922
INFO:rnn:Loss at epoch 45/1000 is 6.226703643798828. Time taken to train = 0.004840139999942039
INFO:rnn:Loss at epoch 46/1000 is 6.142866611480713. Time taken to train = 0.00474860000008448
INFO:rnn:Loss at epoch 47/1000 is 6.17746